# P3 — Validación de Índices Compuestos Tácticos

**Jugador objetivo:** Novak Djokovic  
**Jugador rival:** Taylor Fritz (diestro, sacador agresivo, Top 10 Hard court)  
**Fuente:** JeffSackmann/tennis_MatchChartingProject (CC BY-NC-SA 4.0)  
**Objetivo:** Calcular y validar los 4 índices compuestos del P3 antes de implementarlos en DAX.

**Correcciones v2:**
- Conteo de partidos por superficie filtrado correctamente por Djokovic
- IV BP reemplazado por diferencial de conversión BP W vs L (Opción B)

**Medidas P2 reutilizadas (nombres exactos en Power BI):**
- `% Pts Ganados 1er Saque` → 73.26%
- `% Pts Ganados 2do Saque` → 54.93%
- `% Pts Ganados Al Saque` → 66.89%
- `% Pts Ganados Al Resto` → 40.47%
- `% BP Salvados` → 63.99%
- `% BP Convertidos` → 42.74%
- `% Pts Ganados En Red` → 70.21%
- `% Pts Rally Corto` → 33.90%
- `% Restos Profundos` → 84.17%
- `% Pts Ganados Resto En Juego` → 52.57%

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = r"C:\Users\pater\Desktop\DJOKOVIC\data"

matches      = pd.read_csv(f"{DATA_DIR}/charting-m-matches.csv", encoding="utf-8")
overview     = pd.read_csv(f"{DATA_DIR}/charting-m-stats-Overview.csv", encoding="utf-8")
serve_basics = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ServeBasics.csv", encoding="utf-8")
return_depth = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnDepth.csv", encoding="utf-8")
return_out   = pd.read_csv(f"{DATA_DIR}/charting-m-stats-ReturnOutcomes.csv", encoding="utf-8")
net_points   = pd.read_csv(f"{DATA_DIR}/charting-m-stats-NetPoints.csv", encoding="utf-8")
kp_serve     = pd.read_csv(f"{DATA_DIR}/charting-m-stats-KeyPointsServe.csv", encoding="utf-8")
kp_return    = pd.read_csv(f"{DATA_DIR}/charting-m-stats-KeyPointsReturn.csv", encoding="utf-8")
resultados   = pd.read_csv(f"{DATA_DIR}/djokovic_resultados.csv", encoding="utf-8")

# Limpiar fila corrupta
ID_CORRUPTO = "20240915-M-Davis_Cup_World_Group-RR-Botic_Van_De_Zandschulp-Matteo_Berrettini"
matches = matches[matches["match_id"] != ID_CORRUPTO].copy()
VALID_SURF = ["Clay", "Grass", "Hard"]
matches = matches[matches["Surface"].isin(VALID_SURF)]

# Partidos de Djokovic en matches (para conteo correcto por superficie)
djok_matches = matches[
    (matches["Player 1"] == "Novak Djokovic") |
    (matches["Player 2"] == "Novak Djokovic")
].copy()

# Filtros Djokovic por tabla de hechos
dj_ov  = overview[(overview["player"] == "Novak Djokovic") & (overview["set"] == "Total")]
dj_sb  = serve_basics[serve_basics["player"] == "Novak Djokovic"]
dj_rd  = return_depth[return_depth["player"] == "Novak Djokovic"]
dj_ro  = return_out[return_out["player"] == "Novak Djokovic"]
dj_net = net_points[(net_points["player"] == "Novak Djokovic") & (net_points["row"] == "NetPoints")]
dj_kps = kp_serve[kp_serve["player"] == "Novak Djokovic"]
dj_kpr = kp_return[kp_return["player"] == "Novak Djokovic"]

print("Tablas cargadas correctamente.")
print(f"Partidos Djokovic en matches: {len(djok_matches)}")
print()
print("Distribución correcta por superficie (partidos de Djokovic):")
print(djok_matches["Surface"].value_counts().to_string())

Tablas cargadas correctamente.
Partidos Djokovic en matches: 553

Distribución correcta por superficie (partidos de Djokovic):
Surface
Hard     366
Clay     129
Grass     58


## SECCIÓN 1 — Verificación medidas base P2

In [2]:
pct_1s          = dj_ov["first_won"].sum() / dj_ov["first_in"].sum()
pct_2s          = dj_ov["second_won"].sum() / dj_ov["second_in"].sum()
pct_saque       = (dj_ov["first_won"].sum() + dj_ov["second_won"].sum()) / dj_ov["serve_pts"].sum()
pct_resto       = dj_ov["return_pts_won"].sum() / dj_ov["return_pts"].sum()
pct_bp_salv     = dj_ov["bp_saved"].sum() / dj_ov["bk_pts"].sum()
pct_bp_conv     = dj_kpr[dj_kpr["row"]=="BPO"]["pts_won"].sum() / dj_kpr[dj_kpr["row"]=="BPO"]["pts"].sum()
pct_red         = dj_net["pts_won"].sum() / dj_net["net_pts"].sum()
sb_tot          = dj_sb[dj_sb["row"] == "Total"]
pct_rally_corto = sb_tot["pts_won_lte_3_shots"].sum() / sb_tot["pts"].sum()
rd_tot          = dj_rd[dj_rd["row"] == "Total"]
pct_profundo    = (rd_tot["deep"].sum() + rd_tot["very_deep"].sum()) / \
                  (rd_tot["shallow"].sum() + rd_tot["deep"].sum() + rd_tot["very_deep"].sum())
ro_tot          = dj_ro[dj_ro["row"] == "Total"]
pct_inplay      = ro_tot["in_play_won"].sum() / ro_tot["in_play"].sum()

print("=== VERIFICACIÓN MEDIDAS P2 ===")
for nombre, calc, ref in [
    ("% Pts Ganados 1er Saque",      pct_1s,          0.7326),
    ("% Pts Ganados 2do Saque",      pct_2s,          0.5493),
    ("% Pts Ganados Al Saque",       pct_saque,       0.6689),
    ("% Pts Ganados Al Resto",       pct_resto,       0.4047),
    ("% BP Salvados",                pct_bp_salv,     0.6399),
    ("% BP Convertidos",             pct_bp_conv,     0.4274),
    ("% Pts Ganados En Red",         pct_red,         0.7021),
    ("% Pts Rally Corto",            pct_rally_corto, 0.3390),
    ("% Restos Profundos",           pct_profundo,    0.8417),
    ("% Pts Ganados Resto En Juego", pct_inplay,      0.5257),
]:
    ok = "✓" if abs(calc - ref) < 0.0005 else "⚠ DIFERENCIA"
    print(f"  {ok}  {nombre:<40s}: {calc:.4f}  (ref: {ref:.4f})")

=== VERIFICACIÓN MEDIDAS P2 ===
  ✓  % Pts Ganados 1er Saque                 : 0.7326  (ref: 0.7326)
  ✓  % Pts Ganados 2do Saque                 : 0.5493  (ref: 0.5493)
  ✓  % Pts Ganados Al Saque                  : 0.6689  (ref: 0.6689)
  ✓  % Pts Ganados Al Resto                  : 0.4047  (ref: 0.4047)
  ✓  % BP Salvados                           : 0.6399  (ref: 0.6399)
  ✓  % BP Convertidos                        : 0.4274  (ref: 0.4274)
  ✓  % Pts Ganados En Red                    : 0.7021  (ref: 0.7021)
  ✓  % Pts Rally Corto                       : 0.3390  (ref: 0.3390)
  ✓  % Restos Profundos                      : 0.8417  (ref: 0.8417)
  ✓  % Pts Ganados Resto En Juego            : 0.5257  (ref: 0.5257)


## SECCIÓN 2 — Medidas auxiliares nuevas P3

In [3]:
# % Saque 2S Body
sb_2s       = dj_sb[dj_sb["row"] == "2"]
pct_body_2s = sb_2s["body"].sum() / (sb_2s["wide"].sum() + sb_2s["body"].sum() + sb_2s["t"].sum())

# % Pts Rally Largo (row=9)
ro_9        = dj_ro[dj_ro["row"] == "9"]
pct_largo   = ro_9["pts_won"].sum() / ro_9["pts"].sum()

# % Pts Rally Medio (row=4 y row=6)
ro_medio    = dj_ro[dj_ro["row"].isin(["4", "6"])]
pct_medio   = ro_medio["pts_won"].sum() / ro_medio["pts"].sum()
pct_4       = dj_ro[dj_ro["row"]=="4"]["pts_won"].sum() / dj_ro[dj_ro["row"]=="4"]["pts"].sum()
pct_6       = dj_ro[dj_ro["row"]=="6"]["pts_won"].sum() / dj_ro[dj_ro["row"]=="6"]["pts"].sum()

# Frecuencias para normalización de IA
pts_total_resto  = ro_tot["pts"].sum()
freq_rally_largo = ro_9["pts"].sum() / pts_total_resto
pts_rtotal       = dj_kpr[dj_kpr["row"]=="RTotal"]["pts"].sum()
pts_bpo          = dj_kpr[dj_kpr["row"]=="BPO"]["pts"].sum()
freq_bpo         = pts_bpo / pts_rtotal

# IV BP — Opción B: diferencial conversión BP W vs L
# Calcula % BP Convertidos en victorias y en derrotas por superficie
dj_kpr_res = dj_kpr.merge(resultados[["match_id","Resultado"]], on="match_id", how="inner")
dj_kpr_res = dj_kpr_res[dj_kpr_res["Resultado"].isin(["W","L"])]

bp_W = dj_kpr_res[(dj_kpr_res["row"]=="BPO") & (dj_kpr_res["Resultado"]=="W")]
bp_L = dj_kpr_res[(dj_kpr_res["row"]=="BPO") & (dj_kpr_res["Resultado"]=="L")]
pct_bpo_W = bp_W["pts_won"].sum() / bp_W["pts"].sum()
pct_bpo_L = bp_L["pts_won"].sum() / bp_L["pts"].sum()
diferencial_bp_WL = pct_bpo_W - pct_bpo_L  # validado P2: 44.68% - 31.56% = 13.12pp

print("=== MEDIDAS AUXILIARES NUEVAS P3 ===")
print(f"  % Saque 2S Body                     : {pct_body_2s:.4f}  (espera: 0.3272)")
print(f"  % Pts Rally Largo (row=9)            : {pct_largo:.4f}  (espera: 0.5889)")
print(f"  % Pts Rally Medio (row=4+6 agregado) : {pct_medio:.4f}")
print(f"    └─ row=4 solo                      : {pct_4:.4f}  (espera: 0.3594)")
print(f"    └─ row=6 solo                      : {pct_6:.4f}  (espera: 0.3460)")
print(f"  Freq. puntos 9+ golpes / Total resto : {freq_rally_largo:.4f}")
print(f"  Freq. BPO / RTotal                  : {freq_bpo:.4f}")
print()
print("  IV BP — Opción B (diferencial conversión W vs L):")
print(f"    % BP Convertidos en W              : {pct_bpo_W:.4f}  (ref P2: 0.4468)")
print(f"    % BP Convertidos en L              : {pct_bpo_L:.4f}  (ref P2: 0.3156)")
print(f"    Diferencial W-L                    : {diferencial_bp_WL:.4f}  (ref P2: ~0.131)")

=== MEDIDAS AUXILIARES NUEVAS P3 ===
  % Saque 2S Body                     : 0.3272  (espera: 0.3272)
  % Pts Rally Largo (row=9)            : 0.5889  (espera: 0.5889)
  % Pts Rally Medio (row=4+6 agregado) : 0.3532
    └─ row=4 solo                      : 0.3594  (espera: 0.3594)
    └─ row=6 solo                      : 0.3460  (espera: 0.3460)
  Freq. puntos 9+ golpes / Total resto : 0.2003
  Freq. BPO / RTotal                  : 0.2437

  IV BP — Opción B (diferencial conversión W vs L):
    % BP Convertidos en W              : 0.4468  (ref P2: 0.4468)
    % BP Convertidos en L              : 0.3156  (ref P2: 0.3156)
    Diferencial W-L                    : 0.1311  (ref P2: ~0.131)


## SECCIÓN 3 — Índice de Amenaza (IA)

In [4]:
# IA 1: Saque 2S Predecible
ia_2s_fn   = min(pct_body_2s / 0.50, 1.0)
IA_2s      = round(ia_2s_fn * pct_2s * 1.0, 3)

# IA 2: Rally Largo (9+ golpes)
ia_lg_fn   = min(freq_rally_largo / 0.30, 1.0)
IA_largo   = round(min(ia_lg_fn * pct_largo * 1.2, 1.0), 3)

# IA 3: Conversión Break Points
ia_bp_fn   = min(freq_bpo / 0.35, 1.0)
IA_bp      = round(min(ia_bp_fn * pct_bp_conv * 1.5, 1.0), 3)

# IA 4: Resto Profundo
ia_pr_fn   = min(pct_profundo / 0.90, 1.0)
IA_prof    = round(min(ia_pr_fn * pct_inplay * 1.1, 1.0), 3)

print("=== ÍNDICE DE AMENAZA (IA) ===")
print(f"  IA Resto Profundo         : {IA_prof:.3f}  → MUY ALTA")
print(f"  IA Rally Largo            : {IA_largo:.3f}  → ALTA")
print(f"  IA Conversión BP          : {IA_bp:.3f}  → ALTA")
print(f"  IA Saque 2S Predecible    : {IA_2s:.3f}  → MEDIA")
print()
print("Componentes:")
print(f"  IA Prof: {ia_pr_fn:.3f} × {pct_inplay:.4f} × 1.1 = {IA_prof:.3f}")
print(f"  IA Largo:{ia_lg_fn:.3f} × {pct_largo:.4f} × 1.2 = {IA_largo:.3f}")
print(f"  IA BP:   {ia_bp_fn:.3f} × {pct_bp_conv:.4f} × 1.5 = {IA_bp:.3f}")
print(f"  IA 2S:   {ia_2s_fn:.3f} × {pct_2s:.4f} × 1.0 = {IA_2s:.3f}")

=== ÍNDICE DE AMENAZA (IA) ===
  IA Resto Profundo         : 0.541  → MUY ALTA
  IA Rally Largo            : 0.472  → ALTA
  IA Conversión BP          : 0.446  → ALTA
  IA Saque 2S Predecible    : 0.359  → MEDIA

Componentes:
  IA Prof: 0.935 × 0.5257 × 1.1 = 0.541
  IA Largo:0.668 × 0.5889 × 1.2 = 0.472
  IA BP:   0.696 × 0.4274 × 1.5 = 0.446
  IA 2S:   0.654 × 0.5493 × 1.0 = 0.359


## SECCIÓN 4 — Índice de Vulnerabilidad (IV)

**IV BP — Opción B:** usa el diferencial de conversión BP W vs L como indicador de
cuánto cae Djokovic cuando Fritz genera presión sostenida de break points.

Fórmula IV BP = `diferencial_WL / techo_WL × ajuste_superficie`  
El diferencial es una constante global (no varía por superficie en este cálculo);
el ajuste de superficie refleja la confianza de la muestra.

In [5]:
ajuste_sup = {"Hard": 1.0, "Clay": 0.9, "Grass": 0.6}

# Factor confianza — ahora filtrado correctamente por Djokovic
def factor_conf(n):
    if n >= 300: return 1.0
    if n >= 100: return 0.9
    if n >= 30:  return 0.7
    if n >= 10:  return 0.4
    return 0.3

resultados_iv = {}

for surf, ajuste in ajuste_sup.items():
    # IDs de partidos de Djokovic en esa superficie (correcto)
    ids_surf  = djok_matches[djok_matches["Surface"] == surf]["match_id"]
    n_part    = len(ids_surf)
    fc        = factor_conf(n_part)

    ov_s  = dj_ov[dj_ov["match_id"].isin(ids_surf)]
    sb_s  = dj_sb[dj_sb["match_id"].isin(ids_surf)]
    ro_s  = dj_ro[dj_ro["match_id"].isin(ids_surf)]

    # Medidas base en esa superficie
    pct_saque_s = (ov_s["first_won"].sum() + ov_s["second_won"].sum()) / ov_s["serve_pts"].sum()
    pct_1s_s    = ov_s["first_won"].sum() / ov_s["first_in"].sum()
    pct_2s_s    = ov_s["second_won"].sum() / ov_s["second_in"].sum()
    sb_s_tot    = sb_s[sb_s["row"] == "Total"]
    pct_corto_s = sb_s_tot["pts_won_lte_3_shots"].sum() / sb_s_tot["pts"].sum()
    pct_resto_s = ov_s["return_pts_won"].sum() / ov_s["return_pts"].sum()

    # IV 1: Rally Corto — caída relativa saque global vs rally corto
    IV_corto = round(1.0 * ((pct_saque_s - pct_corto_s) / pct_saque_s) * ajuste, 3)

    # IV 2: Segundo Saque — caída relativa 1S vs 2S
    IV_2s = round(1.0 * ((pct_1s_s - pct_2s_s) / pct_1s_s) * ajuste, 3)

    # IV 3: Rally Medio — caída relativa global resto vs medio (row=4,6)
    ro_m_s   = ro_s[ro_s["row"].isin(["4", "6"])]
    ro_tot_s = ro_s[ro_s["row"] == "Total"]
    pct_med_s  = ro_m_s["pts_won"].sum() / ro_m_s["pts"].sum() if len(ro_m_s) > 0 else np.nan
    pct_res_ro = ro_tot_s["pts_won"].sum() / ro_tot_s["pts"].sum() if len(ro_tot_s) > 0 else np.nan
    IV_medio = round(max(1.0 * ((pct_res_ro - pct_med_s) / pct_res_ro) * ajuste, 0), 3) \
               if not np.isnan(pct_med_s) else 0.0

    # IV 4 — Opción B: diferencial BP conversión W vs L
    # diferencial_bp_WL es constante global; ajuste de superficie aplica confianza
    # Normalizado: diferencial / techo (pct_bpo_W = mejor caso = 1.0 relativo)
    IV_bp = round(fc * (diferencial_bp_WL / pct_bpo_W) * ajuste, 3)

    resultados_iv[surf] = {
        "n_part": n_part, "fc": fc, "ajuste": ajuste,
        "IV_corto": IV_corto, "IV_2s": IV_2s, "IV_medio": IV_medio, "IV_bp": IV_bp,
        "pct_saque": pct_saque_s, "pct_1s": pct_1s_s, "pct_2s": pct_2s_s,
        "pct_corto": pct_corto_s, "pct_med": pct_med_s
    }

print("=== ÍNDICE DE VULNERABILIDAD (IV) POR SUPERFICIE ===")
print(f"{'Vulnerabilidad':<30s} {'Hard':>8s} {'Clay':>8s} {'Grass':>8s}")
print("-" * 56)
for key, lbl in [("IV_corto","IV Rally Corto Saque"),
                 ("IV_2s",   "IV Segundo Saque"),
                 ("IV_medio","IV Rally Medio Resto"),
                 ("IV_bp",   "IV BP Conv. W vs L")]:
    v = [resultados_iv[s][key] for s in ["Hard","Clay","Grass"]]
    print(f"  {lbl:<28s} {v[0]:>8.3f} {v[1]:>8.3f} {v[2]:>8.3f}")

print()
print("Partidos de Djokovic por superficie (correcto):")
for s in ["Hard","Clay","Grass"]:
    r = resultados_iv[s]
    print(f"  {s:<8s}: {r['n_part']:>3d} partidos  (ajuste: {r['ajuste']}  confianza: {r['fc']})")

print()
print("Datos de soporte Hard:")
h = resultados_iv["Hard"]
print(f"  % Pts Al Saque Hard    : {h['pct_saque']:.4f}")
print(f"  % Pts Rally Corto Hard : {h['pct_corto']:.4f}")
print(f"  % Pts 1S Hard          : {h['pct_1s']:.4f}")
print(f"  % Pts 2S Hard          : {h['pct_2s']:.4f}")
print(f"  % Pts Rally Medio Hard : {h['pct_med']:.4f}")
print(f"  Diferencial BP W-L     : {diferencial_bp_WL:.4f}")
print(f"  IV BP (Opción B) Hard  : {h['IV_bp']:.3f}")

=== ÍNDICE DE VULNERABILIDAD (IV) POR SUPERFICIE ===
Vulnerabilidad                     Hard     Clay    Grass
--------------------------------------------------------
  IV Rally Corto Saque            0.482    0.506    0.258
  IV Segundo Saque                0.245    0.209    0.182
  IV Rally Medio Resto            0.133    0.090    0.090
  IV BP Conv. W vs L              0.294    0.238    0.123

Partidos de Djokovic por superficie (correcto):
  Hard    : 366 partidos  (ajuste: 1.0  confianza: 1.0)
  Clay    : 129 partidos  (ajuste: 0.9  confianza: 0.9)
  Grass   :  58 partidos  (ajuste: 0.6  confianza: 0.7)

Datos de soporte Hard:
  % Pts Al Saque Hard    : 0.6754
  % Pts Rally Corto Hard : 0.3497
  % Pts 1S Hard          : 0.7394
  % Pts 2S Hard          : 0.5579
  % Pts Rally Medio Hard : 0.3540
  Diferencial BP W-L     : 0.1311
  IV BP (Opción B) Hard  : 0.294


## SECCIÓN 5 — Prioridad de Acción (PA) — Hard court

In [6]:
fritz_perfil = {
    "Saque agresivo":          0.90,
    "Derecha ofensiva":        0.85,
    "Resto agresivo 2S":       0.80,
    "Punto corto (1-4g)":      0.85,
    "Reves defensivo":         0.55,
    "Juego en red":            0.50,
    "Rally largo (7+ golpes)": 0.40,
}

h  = resultados_iv["Hard"]
fc = h["fc"]

PA_corto = round(h["IV_corto"] * fritz_perfil["Punto corto (1-4g)"]  * fc, 3)
PA_2s    = round(h["IV_2s"]    * fritz_perfil["Resto agresivo 2S"]    * fc, 3)
PA_medio = round(h["IV_medio"] * fritz_perfil["Derecha ofensiva"]     * fc, 3)
PA_bp    = round(h["IV_bp"]    * fritz_perfil["Saque agresivo"]       * fc, 3)

def sem_pa(pa):
    if pa >= 0.35: return "🟢 VERDE  — obligatoria"
    if pa >= 0.15: return "🟡 AMARILLO — situacional"
    return             "🔴 ROJO   — no priorizar"

acciones = [
    ("PA Rally Corto (saque T/Wide agresivo)",   PA_corto),
    ("PA Segundo Saque (resto adelantado)",       PA_2s),
    ("PA Rally Medio (derecha 4-6 golpes)",       PA_medio),
    ("PA BP (saque agresivo en break points)",    PA_bp),
]

print("=== PRIORIDAD DE ACCIÓN (PA) — Hard court ===")
print(f"{'Acción':<45s} {'PA':>6s}  Semáforo")
print("-" * 80)
for n, pa in sorted(acciones, key=lambda x: -x[1]):
    print(f"  {n:<43s} {pa:>6.3f}  {sem_pa(pa)}")

print()
print("INSTRUCCIONES TÁCTICAS PRIORIZADAS:")
print("  🟢 1ª: Servir T (deuce) y Wide (ad) para forzar punto ≤3 golpes")
print("  🟡 2ª: Avanzar dentro de la pista en el segundo saque de Djokovic")
print("  🟡 3ª: Buscar derecha ofensiva en el 4º-6º golpe del rally")
print("  ⚡ BP: Mantener saque agresivo — no cambiar la táctica bajo presión")

=== PRIORIDAD DE ACCIÓN (PA) — Hard court ===
Acción                                            PA  Semáforo
--------------------------------------------------------------------------------
  PA Rally Corto (saque T/Wide agresivo)       0.410  🟢 VERDE  — obligatoria
  PA BP (saque agresivo en break points)       0.265  🟡 AMARILLO — situacional
  PA Segundo Saque (resto adelantado)          0.196  🟡 AMARILLO — situacional
  PA Rally Medio (derecha 4-6 golpes)          0.113  🔴 ROJO   — no priorizar

INSTRUCCIONES TÁCTICAS PRIORIZADAS:
  🟢 1ª: Servir T (deuce) y Wide (ad) para forzar punto ≤3 golpes
  🟡 2ª: Avanzar dentro de la pista en el segundo saque de Djokovic
  🟡 3ª: Buscar derecha ofensiva en el 4º-6º golpe del rally
  ⚡ BP: Mantener saque agresivo — no cambiar la táctica bajo presión


## SECCIÓN 6 — Nivel de Riesgo (NR)

In [7]:
prob_error = {
    "Saque agresivo":    0.15,
    "Resto agresivo 2S": 0.20,
    "Derecha rally 4-6": 0.18,
    "Subida a red":      0.30,
}
coste = {"normal": 1.0, "bp": 1.5, "timing": 1.2, "red": 1.3}

NR_saque_n = round(prob_error["Saque agresivo"]    * coste["normal"], 3)
NR_saque_bp= round(prob_error["Saque agresivo"]    * coste["bp"],     3)
NR_2s      = round(prob_error["Resto agresivo 2S"] * coste["normal"], 3)
NR_derecha = round(prob_error["Derecha rally 4-6"] * coste["timing"], 3)
NR_red     = round(prob_error["Subida a red"]      * coste["red"],    3)

def sem_nr(nr):
    if nr < 0.15:  return "🟢 VERDE  — riesgo bajo"
    if nr < 0.25:  return "🟡 AMARILLO — riesgo medio"
    return             "🔴 ROJO   — riesgo alto"

niveles = [
    ("NR Saque agresivo (normal)",  NR_saque_n),
    ("NR Saque agresivo (BP)",      NR_saque_bp),
    ("NR Resto agresivo 2S",        NR_2s),
    ("NR Derecha rally 4-6 golpes", NR_derecha),
    ("NR Subida a red",             NR_red),
]

print("=== NIVEL DE RIESGO (NR) ===")
print(f"{'Acción':<35s} {'NR':>6s}  Semáforo")
print("-" * 70)
for n, nr in sorted(niveles, key=lambda x: -x[1]):
    print(f"  {n:<33s} {nr:>6.3f}  {sem_nr(nr)}")

print()
print(f"Referencia: Djokovic gana {pct_red:.1%} de los puntos en red →",
      f"NR subida a red = {NR_red:.3f} (ROJO justificado)")

=== NIVEL DE RIESGO (NR) ===
Acción                                  NR  Semáforo
----------------------------------------------------------------------
  NR Subida a red                    0.390  🔴 ROJO   — riesgo alto
  NR Saque agresivo (BP)             0.225  🟡 AMARILLO — riesgo medio
  NR Derecha rally 4-6 golpes        0.216  🟡 AMARILLO — riesgo medio
  NR Resto agresivo 2S               0.200  🟡 AMARILLO — riesgo medio
  NR Saque agresivo (normal)         0.150  🟡 AMARILLO — riesgo medio

Referencia: Djokovic gana 70.2% de los puntos en red → NR subida a red = 0.390 (ROJO justificado)


## SECCIÓN 7 — Tabla consolidada final

In [8]:
h = resultados_iv["Hard"]

filas = [
    ("Resto profundo (Djokovic)",     IA_prof,  None,         None,     None),
    ("Rally largo 9+ golpes",         IA_largo, None,         None,     None),
    ("Conversión BP (Djokovic)",      IA_bp,    h["IV_bp"],   PA_bp,    NR_saque_bp),
    ("Saque 2S predecible",           IA_2s,    h["IV_2s"],   PA_2s,    NR_2s),
    ("Rally corto ≤3g (Fritz)",       None,     h["IV_corto"],PA_corto, NR_saque_n),
    ("Rally medio 4-6g",              None,     h["IV_medio"],PA_medio, NR_derecha),
    ("Subida a red",                  None,     None,         None,     NR_red),
]

def fmt(v): return f"{v:.3f}" if v is not None else "  — "

print("=== TABLA CONSOLIDADA DE ÍNDICES — Hard court ===")
print(f"{'Patrón':<35s} {'IA':>6s} {'IV':>6s} {'PA':>6s} {'NR':>6s}")
print("-" * 63)
for patron, ia, iv, pa, nr in filas:
    print(f"  {patron:<33s} {fmt(ia):>6s} {fmt(iv):>6s} {fmt(pa):>6s} {fmt(nr):>6s}")

print()
print("LEYENDA:")
print("  IA = Índice Amenaza      [0-1]: daño de Djokovic con ese patrón")
print("  IV = Índice Vulnerabilidad[0-1]: ventana explotable por Fritz")
print("  PA = Prioridad de Acción [0-1]: qué debe hacer Fritz primero")
print("  NR = Nivel de Riesgo     [0-1]: coste de equivocarse")
print("  Semáforo PA: 🟢 ≥0.35 | 🟡 0.15–0.35 | 🔴 <0.15")
print("  Semáforo NR: 🟢 <0.15 | 🟡 0.15–0.25 | 🔴 >0.25")

=== TABLA CONSOLIDADA DE ÍNDICES — Hard court ===
Patrón                                  IA     IV     PA     NR
---------------------------------------------------------------
  Resto profundo (Djokovic)          0.541     —      —      — 
  Rally largo 9+ golpes              0.472     —      —      — 
  Conversión BP (Djokovic)           0.446  0.294  0.265  0.225
  Saque 2S predecible                0.359  0.245  0.196  0.200
  Rally corto ≤3g (Fritz)               —   0.482  0.410  0.150
  Rally medio 4-6g                      —   0.133  0.113  0.216
  Subida a red                          —      —      —   0.390

LEYENDA:
  IA = Índice Amenaza      [0-1]: daño de Djokovic con ese patrón
  IV = Índice Vulnerabilidad[0-1]: ventana explotable por Fritz
  PA = Prioridad de Acción [0-1]: qué debe hacer Fritz primero
  NR = Nivel de Riesgo     [0-1]: coste de equivocarse
  Semáforo PA: 🟢 ≥0.35 | 🟡 0.15–0.35 | 🔴 <0.15
  Semáforo NR: 🟢 <0.15 | 🟡 0.15–0.25 | 🔴 >0.25


## SECCIÓN 8 — IV por superficie (tabla completa)

In [9]:
print("=== IV POR SUPERFICIE ===")
print(f"{'IV':<30s} {'Hard':>8s} {'Clay':>8s} {'Grass':>8s}  Nota")
print("-" * 78)
for key, lbl, nota in [
    ("IV_corto", "IV Rally Corto Saque",  "Alta en Hard y Clay; baja en Grass (bote bajo)"),
    ("IV_2s",    "IV Segundo Saque",      "Consistente entre superficies"),
    ("IV_medio", "IV Rally Medio Resto",  "Menor en Clay/Grass — Djokovic más cómodo"),
    ("IV_bp",    "IV BP Conv. W vs L",    "Grass reducido por muestra pequeña (58p)"),
]:
    v = [resultados_iv[s][key] for s in ["Hard","Clay","Grass"]]
    print(f"  {lbl:<28s} {v[0]:>8.3f} {v[1]:>8.3f} {v[2]:>8.3f}  {nota}")

print()
print("Partidos de Djokovic por superficie:")
for s in ["Hard","Clay","Grass"]:
    r = resultados_iv[s]
    print(f"  {s:<8s}: {r['n_part']:>3d} partidos  "
          f"(ajuste superficie: {r['ajuste']}  factor confianza: {r['fc']})")

=== IV POR SUPERFICIE ===
IV                                 Hard     Clay    Grass  Nota
------------------------------------------------------------------------------
  IV Rally Corto Saque            0.482    0.506    0.258  Alta en Hard y Clay; baja en Grass (bote bajo)
  IV Segundo Saque                0.245    0.209    0.182  Consistente entre superficies
  IV Rally Medio Resto            0.133    0.090    0.090  Menor en Clay/Grass — Djokovic más cómodo
  IV BP Conv. W vs L              0.294    0.238    0.123  Grass reducido por muestra pequeña (58p)

Partidos de Djokovic por superficie:
  Hard    : 366 partidos  (ajuste superficie: 1.0  factor confianza: 1.0)
  Clay    : 129 partidos  (ajuste superficie: 0.9  factor confianza: 0.9)
  Grass   :  58 partidos  (ajuste superficie: 0.6  factor confianza: 0.7)
